<a href="https://colab.research.google.com/github/arunangshu19/PINN/blob/schrodinger/qho_n_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

# ===============================
# Setup
torch.manual_seed(0)
np.random.seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

n = 1
E_n = (2*n+1)/2
domain = [-10.0, 10.0]
N_grid = 15000
batch_size = 1024
epochs_stage1 = 5000
epochs_stage2 = 5000
lambda_bc = 10.0
N_runs = 5

# ===============================
def true_psi_n(x, n):
    Hn = [torch.ones_like(x), 2*x]
    for k in range(2, n+1):
        Hn.append(2*x*Hn[-1] - 2*(k-1)*Hn[-2])
    return torch.exp(-x**2/2) * Hn[n]

class FCNN(nn.Module):
    def __init__(self, layers):
        super().__init__()
        net=[]
        for i in range(len(layers)-2):
            net.append(nn.Linear(layers[i], layers[i+1]))
            net.append(nn.Tanh())
        net.append(nn.Linear(layers[-2], layers[-1]))
        self.net = nn.Sequential(*net)
    def forward(self,x): return self.net(x)

def pde_residual(x, model):
    x = x.requires_grad_(True)
    psi = model(x)
    dpsi = torch.autograd.grad(psi, x, grad_outputs=torch.ones_like(psi), create_graph=True)[0]
    d2psi = torch.autograd.grad(dpsi, x, grad_outputs=torch.ones_like(dpsi), create_graph=True)[0]
    V = 0.5*x**2
    return -0.5*d2psi + V*psi - E_n*psi

def bc_loss(model):
    xb = torch.tensor([[domain[0]],[domain[1]]], device=device)
    psib = model(xb)
    return torch.mean(psib**2)

def train_two_stage():
    model = FCNN([1,128,128,128,1]).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    X_grid = torch.linspace(domain[0], domain[1], N_grid).view(-1,1).to(device)
    loss_log = []

    # Stage 1: PDE only
    for ep in range(epochs_stage1):
        idx = torch.randint(0, N_grid, (batch_size,), device=device)
        x_f = X_grid[idx]
        res = pde_residual(x_f, model)
        loss = torch.mean(res**2)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        loss_log.append(loss.item())

    # Stage 2: PDE + BC
    for ep in range(epochs_stage2):
        idx = torch.randint(0, N_grid, (batch_size,), device=device)
        x_f = X_grid[idx]
        res = pde_residual(x_f, model)
        loss = torch.mean(res**2) + lambda_bc * bc_loss(model)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        loss_log.append(loss.item())

    with torch.no_grad():
        psi_pred = model(X_grid)
    return X_grid.detach().cpu(), psi_pred.detach().cpu(), np.array(loss_log)

# ===============================
psi_all, loss_all = [], []
for _ in range(N_runs):
    xg, yp, ll = train_two_stage()
    pred = yp.squeeze().numpy()
    maxabs = np.max(np.abs(pred))
    if maxabs > 0: pred = pred/maxabs
    if pred[np.argmax(np.abs(pred))] < 0: pred *= -1
    psi_all.append(pred); loss_all.append(ll)

psi_all = np.stack(psi_all)
loss_all = np.stack(loss_all)
mean_psi = np.mean(psi_all, axis=0)
std_psi = np.std(psi_all, axis=0)
x_plot = xg.squeeze().numpy()
true_psi = true_psi_n(torch.tensor(x_plot).view(-1,1), n).squeeze()
true_psi = true_psi / torch.max(torch.abs(true_psi))

# ===============================
plt.figure(figsize=(10,4))
plt.plot(x_plot, mean_psi, 'b', label='Mean Prediction')
# plt.fill_between(x_plot, mean_psi-std_psi, mean_psi+std_psi,
#                  color='blue', alpha=0.3, label='±1 Std')
plt.plot(x_plot, true_psi.numpy(), 'r--', label='True ψ₁(x)')
plt.title("2-Stage Training: Mean vs True")
plt.xlabel("x"); plt.ylabel("ψ(x)"); plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()

plt.figure(figsize=(10,4))
for i in range(N_runs):
    plt.plot(x_plot, psi_all[i], color='gray', alpha=0.4)
plt.plot(x_plot, mean_psi, 'b', label='Mean Prediction')
plt.fill_between(x_plot, mean_psi-std_psi, mean_psi+std_psi,
                 color='blue', alpha=0.3,  label='±1 Std')
# plt.plot(x_plot, true_psi.numpy(), 'r--', label='True ψ₁(x)')
plt.title("2-Stage Training: All Runs with Mean ± Std")
plt.xlabel("x"); plt.ylabel("ψ(x)"); plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()

plt.figure(figsize=(6,4))
for i in range(N_runs):
    plt.plot(np.log10(loss_all[i]+1e-10), alpha=0.7)
plt.title("2-Stage Training: Log-Scale Loss")
plt.xlabel("Iteration"); plt.ylabel("log10(Loss)"); plt.grid(True); plt.tight_layout(); plt.show()

In [ ]:
# # ===============================
# # Plot 1: Log-loss for each run
# plt.figure(figsize=(6,4))
# for i in range(N_runs):
#     plt.plot(np.log10(loss_all[i] + 1e-12), alpha=0.7, label=f"Run {i+1}")
# plt.title("2-Stage Training: Log-Scale Loss")
# plt.xlabel("Iteration"); plt.ylabel("log10(Loss)")
# plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()

# # ===============================
# # Plot 2: Normalized predictions vs true solution
# plt.figure(figsize=(10,4))
# for i in range(N_runs):
#     plt.plot(x_plot, psi_all[i], color='gray', alpha=0.4)
# plt.plot(x_plot, true_psi.numpy(), 'r--', linewidth=2, label='True ψ₁(x)')
# plt.title("Normalized Predictions vs True Solution")
# plt.xlabel("x"); plt.ylabel("ψ(x)")
# plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()

# # ===============================
# # Plot 3: Unnormalized predictions with mean ± std
# # (recompute without normalizing each run!)
# psi_all_raw = []
# for _ in range(N_runs):
#     xg, yp, ll = train_two_stage()
#     psi_all_raw.append(yp.squeeze().numpy())
# psi_all_raw = np.stack(psi_all_raw)

# mean_raw = np.mean(psi_all_raw, axis=0)
# std_raw = np.std(psi_all_raw, axis=0)

# plt.figure(figsize=(10,4))
# plt.plot(x_plot, mean_raw, 'b', label='Mean (unnormalized)')
# plt.fill_between(x_plot, mean_raw-std_raw, mean_raw+std_raw,
#                  color='blue', alpha=0.3, label='±1 Std')
# plt.title("Unnormalized Predictions: Mean ± Std")
# plt.xlabel("x"); plt.ylabel("ψ(x)")
# plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()
# ===============================
# Plot 1: Log-loss for each run
plt.figure(figsize=(6,4))
for i in range(N_runs):
    plt.plot(np.log10(loss_all[i] + 1e-12), alpha=0.7, label=f"Run {i+1}")
plt.title("2-Stage Training: Log-Scale Loss")
plt.xlabel("Iteration"); plt.ylabel("log10(Loss)")
plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()

# ===============================
# Plot 2: Normalized mean prediction vs true solution
# (normalize the MEAN, not each run)
mean_pred = np.mean(psi_all, axis=0)
mean_pred = mean_pred / np.max(np.abs(mean_pred))   # normalize once
if mean_pred[np.argmax(np.abs(mean_pred))] < 0:
    mean_pred *= -1

plt.figure(figsize=(10,4))
plt.plot(x_plot, mean_pred, 'b', label='Mean Prediction (normalized)')
plt.plot(x_plot, true_psi.numpy(), 'r--', linewidth=2, label='True ψ₁(x)')
plt.title("Normalized Mean Prediction vs True Solution")
plt.xlabel("x"); plt.ylabel("ψ(x)")
plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()

# ===============================
# Plot 3: Unnormalized predictions with mean ± std
psi_all_raw = []
for _ in range(N_runs):
    xg, yp, ll = train_two_stage()
    psi_all_raw.append(yp.squeeze().numpy())
psi_all_raw = np.stack(psi_all_raw)

mean_raw = np.mean(psi_all_raw, axis=0)
std_raw = np.std(psi_all_raw, axis=0)

plt.figure(figsize=(10,4))
plt.plot(x_plot, mean_raw, 'b', label='Mean (unnormalized)')
plt.fill_between(x_plot, mean_raw-std_raw, mean_raw+std_raw,
                 color='blue', alpha=0.3, label='±1 Std')
plt.title("Unnormalized Predictions: Mean ± Std")
plt.xlabel("x"); plt.ylabel("ψ(x)")
plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()
